# Imports

In [1]:
import xarray as xr
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np
from scipy.stats import t

# PBSCLuster

In [2]:
# cluster = PBSCluster(
#     cores=1, # The number of cores you want
#     memory='64GB', # Amount of memory
#     processes=1, # How many processes
#     queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
#     local_directory='$TMPDIR', # Use your local directory
#     resource_spec='select=1:ncpus=1:mem=64GB', # Specify resources
#     account='P93300313', # Input your project ID here
#     walltime='01:00:00', # Amount of wall time
#     n_workers=4,
# )
# # Setup your client
# client = Client(cluster)

/glade/u/home/acruz/.conda/envs/EOFa_2025/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46199 instead
  warnings.warn(


In [27]:
# client#.shutdown()

# Function

In [2]:
def xr_regression(x, y, lag_x=0, lag_y=0, dim="time", alternative="two-sided"):
    """
    From https://stackoverflow.com/questions/52108417/how-to-apply-linear-regression-to-every-pixel-in-a-large-multi-dimensional-array
    requires scipy.stats as t
    Takes two xr.Datarrays of any dimensions (input data could be a 1D
    time series, or for example, have three dimensions e.g. time, lat,
    lon), and returns covariance, correlation, coefficient of
    determination, regression slope, intercept, p-value and standard
    error, and number of valid observations (n) between the two datasets
    along their aligned first dimension.

    Datasets can be provided in any order, but note that the regression
    slope and intercept will be calculated for y with respect to x.

    Inspired by:
    https://hrishichandanpurkar.blogspot.com/2017/09/vectorized-functions-for-correlation.html

    Parameters
    ----------
    x, y : xarray DataArray
        Two xarray DataArrays with any number of dimensions, both
        sharing the same first dimension
    lag_x, lag_y : int, optional
        Optional integers giving lag values to assign to either of the
        data, with lagx shifting x, and lagy shifting y with the
        specified lag amount.
    dim : str, optional
        An optional string giving the name of the dimension on which to
        align (and optionally lag) datasets. The default is 'time'.
    alternative : string, optional
        Defines the alternative hypothesis. Default is 'two-sided'.
        The following options are available:

        * 'two-sided': slope of the regression line is nonzero
        * 'less': slope of the regression line is less than zero
        * 'greater':  slope of the regression line is greater than zero

    Returns
    -------
    regression_ds : xarray.Dataset
        A dataset comparing the two input datasets along their aligned
        dimension, containing variables including covariance, correlation,
        coefficient of determination, regression slope, intercept,
        p-value and standard error, and number of valid observations (n).

    """

    # Shift x and y data if lags are specified
    if lag_x != 0:
        # If x lags y by 1, x must be shifted 1 step backwards. But as
        # the 'zero-th' value is nonexistant, xarray assigns it as
        # invalid (nan). Hence it needs to be dropped
        x = x.shift(**{dim: -lag_x}).dropna(dim=dim)

        # Next re-align the two datasets so that y adjusts to the
        # changed coordinates of x
        x, y = xr.align(x, y)

    if lag_y != 0:
        y = y.shift(**{dim: -lag_y}).dropna(dim=dim)

    # Ensure that the data are properly aligned to each other.
    x, y = xr.align(x, y)

    # Compute data length, mean and standard deviation along dim
    n = y.notnull().sum(dim=dim)
    xmean = x.mean(dim=dim)
    ymean = y.mean(dim=dim)
    xstd = x.std(dim=dim)
    ystd = y.std(dim=dim)

    # Compute covariance, correlation and coefficient of determination
    cov = ((x - xmean) * (y - ymean)).sum(dim=dim) / (n)
    cor = cov / (xstd * ystd)
    r2 = cor**2

    # Compute regression slope and intercept
    slope = cov / (xstd**2)
    intercept = ymean - xmean * slope

    # Compute t-statistics and standard error
    tstats = cor * np.sqrt(n - 2) / np.sqrt(1 - cor**2)
    stderr = slope / tstats

    # Calculate p-values for different alternative hypotheses.
    if alternative == "two-sided":
        pval = t.sf(np.abs(tstats), n - 2) * 2
    elif alternative == "greater":
        pval = t.sf(tstats, n - 2)
    elif alternative == "less":
        pval = t.cdf(np.abs(tstats), n - 2)

    # Wrap p-values into an xr.DataArray
    pval = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    # Combine into single dataset
    regression_ds = xr.merge(
        [
            cov.rename("cov").astype(np.float32),
            cor.rename("cor").astype(np.float32),
            r2.rename("r2").astype(np.float32),
            slope.rename("slope").astype(np.float32),
            intercept.rename("intercept").astype(np.float32),
            pval.rename("pvalue").astype(np.float32),
            stderr.rename("stderr").astype(np.float32),
            n.rename("n").astype(np.int16),
        ]
    )

    return regression_ds

# Data Import

In [3]:
EOF_ds = xr.open_dataset("/glade/work/acruz/Reanalysis/HadISST_CANI_EANI.nc")
EOF_ds

<xarray.Dataset> Size: 29kB
Dimensions:  (time: 1212)
Coordinates:
  * time     (time) datetime64[ns] 10kB 1914-01-16T12:00:00 ... 2014-12-16T12...
Data variables:
    EANI     (time) float64 10kB ...
    CANI     (time) float64 10kB ...
Attributes:
    title:        Atlantic Niño Index from HadISST
    description:  Eastern and Central Atlantic Niño Index from EOFa done on H...
    created:      2025-10-17

In [4]:
UW_ds = xr.open_dataset('/glade/work/acruz/Reanalysis/ERA5_UW_tropic_anom.nc')
UW_ds

<xarray.Dataset> Size: 125MB
Dimensions:    (time: 528, level: 37, longitude: 801)
Coordinates:
  * time       (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2022-12-01
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * longitude  (longitude) float64 6kB -180.0 -179.8 -179.5 ... 19.5 19.75 20.0
    month      (time) int64 4kB ...
Data variables:
    U          (time, level, longitude) float32 63MB ...
    utc_date   (time) float64 4kB ...
    W          (time, level, longitude) float32 63MB ...

# Select Data

In [5]:
dates = slice('1914-01-01', '2014-12-31')
UW_ds = UW_ds.sel(time=dates)
EOF_ds = EOF_ds.sel(time=dates)
EOF_ds

<xarray.Dataset> Size: 29kB
Dimensions:  (time: 1212)
Coordinates:
  * time     (time) datetime64[ns] 10kB 1914-01-16T12:00:00 ... 2014-12-16T12...
Data variables:
    EANI     (time) float64 10kB ...
    CANI     (time) float64 10kB ...
Attributes:
    title:        Atlantic Niño Index from HadISST
    description:  Eastern and Central Atlantic Niño Index from EOFa done on H...
    created:      2025-10-17

# Seasonal Peak Mean

In [6]:
U_JJA = UW_ds['U'].sel(time=UW_ds.time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
W_JJA = UW_ds['W'].sel(time=UW_ds.time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()

U_SON = UW_ds['U'].sel(time=UW_ds.time.dt.month.isin([9, 10, 11])).groupby('time.year').mean()
W_SON = UW_ds['W'].sel(time=UW_ds.time.dt.month.isin([9, 10, 11])).groupby('time.year').mean()

EANI = EOF_ds['EANI'].sel(time=EOF_ds['EANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
CANI = EOF_ds['CANI'].sel(time=EOF_ds['CANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()

In [7]:
EANI

<xarray.DataArray 'EANI' (year: 101)> Size: 808B
array([-1.10461447, -0.03364281,  0.39418698, -0.26569131, -1.59333535,
       -0.50258284,  0.00498354,  2.06026045, -0.81339959, -1.00260113,
        0.9715834 ,  0.6874735 ,  0.87966507,  0.3725751 ,  0.45111064,
       -0.70002795, -0.23055106,  0.16308413, -0.68842862,  0.94580698,
        1.82712391,  0.03119553, -0.87397099,  2.04974887,  0.9710414 ,
        1.43233977, -0.80620474, -0.28780053,  0.12419661, -0.43032913,
        2.63657961, -0.16121765, -1.31217002, -0.23504968, -1.13841612,
        1.19608206, -0.04917973,  0.64656083,  0.73474528,  0.66985428,
       -0.89458831, -0.25513578, -1.0827241 , -0.67235468, -2.29788818,
       -0.13634646,  0.64530592, -0.14797946, -0.25223862,  2.59714457,
       -0.94222623, -0.56265921,  1.04267866, -1.24713094,  0.34138982,
       -0.63266642, -0.12691344, -0.18624189, -1.10789441,  1.08935319,
        0.06148212, -0.67393514, -2.33544621, -0.52360564, -1.67252949,
       -0.25071523, -0.94858928, -0.81288438, -2.40301366, -1.6556293 ,
        2.11537108, -0.36941375, -0.52130439,  1.44674699,  1.77087298,
        0.72744732, -0.34875013,  0.30752057, -1.88102779, -0.21349308,
       -1.15662421,  0.96817386,  0.77690615, -0.85625102,  1.80547381,
        1.26382628, -0.128568  , -0.00385357,  0.34179684,  0.82374667,
       -0.21367135, -0.49219619,  0.74396144,  0.88774818,  1.14918162,
       -0.01773452,  0.90155555, -0.32642032, -0.19125816, -0.26453539,
       -0.11094277])
Coordinates:
  * year     (year) int64 808B 1914 1915 1916 1917 1918 ... 2011 2012 2013 2014
Attributes:
    description:  Eastern Atlantic Niño Index. PCs scaled by STD and roll mea...

# Regression

In [8]:
U_EANI_JJAr = xr_regression(EANI, U_JJA, dim='year')
W_EANI_JJAr = xr_regression(EANI, W_JJA, dim='year')

In [9]:
U_CANI_JJAr = xr_regression(CANI, U_JJA, dim='year')
W_CANI_JJAr = xr_regression(CANI, W_JJA, dim='year')

In [10]:
U_EANI_SONr = xr_regression(EANI, U_SON, dim='year')
W_EANI_SONr = xr_regression(EANI, W_SON, dim='year')

In [11]:
U_CANI_SONr = xr_regression(CANI, U_SON, dim='year')
W_CANI_SONr = xr_regression(CANI, W_SON, dim='year')

In [12]:
U_EANI_JJAr

<xarray.Dataset> Size: 896kB
Dimensions:    (level: 37, longitude: 801)
Coordinates:
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * longitude  (longitude) float64 6kB -180.0 -179.8 -179.5 ... 19.5 19.75 20.0
Data variables:
    cov        (level, longitude) float32 119kB 0.5545 0.552 ... -0.03479
    cor        (level, longitude) float32 119kB 0.0669 0.06657 ... -0.4891
    r2         (level, longitude) float32 119kB 0.004476 0.004432 ... 0.2392
    slope      (level, longitude) float32 119kB 0.5306 0.5282 ... -0.03329
    intercept  (level, longitude) float32 119kB 0.6582 0.6584 ... 0.006974
    pvalue     (level, longitude) float32 119kB 0.6983 0.6997 ... 0.002471
    stderr     (level, longitude) float32 119kB 1.357 1.358 ... 0.01092 0.01018
    n          (level, longitude) int16 59kB 36 36 36 36 36 ... 36 36 36 36 36
Attributes:
    description:  Eastern Atlantic Niño Index. PCs scaled by STD and roll mea...

# Export

In [13]:
regression = xr.Dataset(
    data_vars={
        "JJA_U_EANI_slope" :(['level', 'lon'], U_EANI_JJAr['slope'].data),
        "JJA_U_EANI_pvalue":(['level', 'lon'], U_EANI_JJAr['pvalue'].data),
        "JJA_W_EANI_slope" :(['level', 'lon'], W_EANI_JJAr['slope'].data),
        "JJA_W_EANI_pvalue":(['level', 'lon'], W_EANI_JJAr['pvalue'].data),
        "JJA_U_CANI_slope" :(['level', 'lon'], U_CANI_JJAr['slope'].data),
        "JJA_U_CANI_pvalue":(['level', 'lon'], U_CANI_JJAr['pvalue'].data),
        "JJA_W_CANI_slope" :(['level', 'lon'], W_CANI_JJAr['slope'].data),
        "JJA_W_CANI_pvalue":(['level', 'lon'], W_CANI_JJAr['pvalue'].data),
        
        "SON_U_EANI_slope" :(['level', 'lon'], U_EANI_SONr['slope'].data),
        "SON_U_EANI_pvalue":(['level', 'lon'], U_EANI_SONr['pvalue'].data),
        "SON_W_EANI_slope" :(['level', 'lon'], W_EANI_SONr['slope'].data),
        "SON_W_EANI_pvalue":(['level', 'lon'], W_EANI_SONr['pvalue'].data),
        "SON_U_CANI_slope" :(['level', 'lon'], U_CANI_SONr['slope'].data),
        "SON_U_CANI_pvalue":(['level', 'lon'], U_CANI_SONr['pvalue'].data),
        "SON_W_CANI_slope" :(['level', 'lon'], W_CANI_SONr['slope'].data),
        "SON_W_CANI_pvalue":(['level', 'lon'], W_CANI_SONr['pvalue'].data),
    },
   coords={
       "lon": U_EANI_JJAr['longitude'].data,
       "level": U_EANI_JJAr['level'].data
   },
    attrs={
        "Description": "Linear regression results from ERA5 wind anomalies during JJA and SON against AN peak in JJA"
    }
)
regression

<xarray.Dataset> Size: 2MB
Dimensions:            (level: 37, lon: 801)
Coordinates:
  * level              (level) float64 296B 1.0 2.0 3.0 ... 950.0 975.0 1e+03
  * lon                (lon) float64 6kB -180.0 -179.8 -179.5 ... 19.75 20.0
Data variables: (12/16)
    JJA_U_EANI_slope   (level, lon) float32 119kB 0.5306 0.5282 ... -0.03329
    JJA_U_EANI_pvalue  (level, lon) float32 119kB 0.6983 0.6997 ... 0.002471
    JJA_W_EANI_slope   (level, lon) float32 119kB 2.511e-06 ... 9.846e-05
    JJA_W_EANI_pvalue  (level, lon) float32 119kB 0.07477 0.07674 ... 0.3439
    JJA_U_CANI_slope   (level, lon) float32 119kB 0.736 0.7339 ... -0.02861
    JJA_U_CANI_pvalue  (level, lon) float32 119kB 0.622 0.6231 ... 0.02038
    ...                 ...
    SON_W_EANI_slope   (level, lon) float32 119kB 2.089e-08 ... -4.079e-05
    SON_W_EANI_pvalue  (level, lon) float32 119kB 0.9789 0.9752 ... 0.5531
    SON_U_CANI_slope   (level, lon) float32 119kB 0.6391 0.6391 ... 0.001256
    SON_U_CANI_pvalue  (level, lon) float32 119kB 0.7116 0.7117 ... 0.8655
    SON_W_CANI_slope   (level, lon) float32 119kB 4.599e-07 ... -9.486e-05
    SON_W_CANI_pvalue  (level, lon) float32 119kB 0.5922 0.5749 ... 0.1493 0.202
Attributes:
    Description:  Linear regression results from ERA5 wind anomalies during J...

In [15]:
regression.to_netcdf('/glade/work/acruz/Reanalysis/wind_regression.nc')